# Reproducing, from scratch: *Social sentiment tone shows no detectable lead over short-horizon price direction*

This notebook **re-implements the method of the Instrumetriq direction note step by step** - it does not call a
pre-written script. Every quantity is built here from the raw dataset fields, so you can see which field is read,
how it is paired with the forward return, and the arithmetic behind each number.

It uses only the **free weekly samples** (one Tier 3 day per week). We clone the public repo solely to obtain the
sample `.parquet` files; all computation happens below.

**The question.** Does the *tone* of a coin's social posts - how bullish or bearish they are - carry information
about the coin's subsequent price *direction*? We compute the Spearman rank correlation (information coefficient,
IC) between tone and the signed forward return, with a day-level bootstrap interval. A directional signal would
show a materially non-zero IC at a no-look-ahead horizon (k >= 0).

**Expected result: a bounded null.** On the free samples the forward ICs straddle zero (|IC| well under
0.03, the rough floor of a usable directional signal). See the note for the load-bearing caveat: the negative-tone
channel is only ~64% complete, so a weak bearish signal could be partially masked.

## 0. Setup

In [ ]:
!pip -q install pandas pyarrow numpy
import glob
import numpy as np
import pandas as pd

## 1. Get the sample data (only the parquet files)

Anonymous read of a **public** repo - no login or authorization needed.

In [ ]:
!git clone --depth 1 https://github.com/SiCkGFX/instrumetriq-public.git 2>/dev/null || (cd instrumetriq-public && git pull -q)
FILES = sorted(glob.glob('instrumetriq-public/samples/week_*/*_tier3.parquet'))
print(len(FILES), 'weekly Tier 3 sample files')

## 2. What one record looks like

Each row is one coin over one ~2-hour session. We use two nested columns:

- `twitter_sentiment_windows.last_cycle.hybrid_decision_stats.mean_score` - the **tone** at session admission,
  the mean model score over the coin's scored posts (negative = bearish, positive = bullish). Fixed before the
  price path that follows, so there is no look-ahead.
- `spot_prices` - the ~700 price ticks that make up the session (our **outcome**).

Let's print the exact paths from one record.

In [ ]:
df0 = pd.read_parquet(FILES[0], columns=['symbol','snapshot_ts','twitter_sentiment_windows','spot_prices'])
r   = df0.iloc[0]
lc  = r['twitter_sentiment_windows'].get('last_cycle')
hds = lc.get('hybrid_decision_stats') or {}
sp  = r['spot_prices']
print('symbol            :', r['symbol'])
print('snapshot_ts       :', r['snapshot_ts'])
print('mean_score (tone) :', hds.get('mean_score'),
      '  <- twitter_sentiment_windows.last_cycle.hybrid_decision_stats.mean_score')
print('pos_ratio         :', hds.get('pos_ratio'),
      '  <- ...hybrid_decision_stats.pos_ratio')
print('posts_total       :', lc.get('posts_total'))
print('price ticks       :', len(sp), ' first/last mid:', sp[0]['mid'], '/', sp[-1]['mid'])

## 3. Extract tone and the signed return per session

Keep sessions with observable, scored activity (`posts_total > 0`, not `is_silent`, a non-null `mean_score`).
The session's **signed return** is `mid_last / mid_first - 1` on every 3rd tick (~30 s spacing).

In [ ]:
def extract(files):
    rows = []
    for fp in files:
        df = pd.read_parquet(fp, columns=['symbol','snapshot_ts',
                                          'twitter_sentiment_windows','spot_prices'])
        for sym, ts, tsw, sp in zip(df['symbol'].values, df['snapshot_ts'].values,
                                    df['twitter_sentiment_windows'].values, df['spot_prices'].values):
            lc = tsw.get('last_cycle') if hasattr(tsw, 'get') else None
            if lc is None:
                continue
            posts  = lc.get('posts_total')
            silent = (lc.get('sentiment_activity') or {}).get('is_silent')
            hds    = lc.get('hybrid_decision_stats') or {}
            ms, pr = hds.get('mean_score'), hds.get('pos_ratio')
            if not posts or silent or ms is None:
                continue
            if sp is None or len(sp) < 6:
                continue
            mids = [float(s['mid']) for s in sp[::3] if s.get('mid')]
            if len(mids) < 5 or not mids[0]:
                continue
            sret = mids[-1]/mids[0] - 1                 # signed session return
            rows.append((sym, pd.Timestamp(ts), float(ms),
                         float(pr) if pr is not None else np.nan, sret))
    d = pd.DataFrame(rows, columns=['symbol','ts','mean_score','pos_ratio','sret'])
    d['ts']  = pd.to_datetime(d['ts'], utc=True)
    d['day'] = d['ts'].dt.strftime('%Y-%m-%d')
    return d.sort_values(['symbol','ts'])

d = extract(FILES)
print(f'{len(d):,} scored sessions across {d.symbol.nunique()} coins')
d.head()

## 4. Pair tone with the *forward* return (within-Sunday)

To test for a **lead**, pair each session's tone with the signed return of a session that comes **later** for the
same coin. On the full contiguous archive this is simply the coin's next session(s). The weekly samples are one
non-contiguous Sunday per week, so a cross-day 'next session' does not exist - but within a single Sunday a coin
has several same-day sessions, so we pair within `(symbol, day)`. This is the direction-side analog of the
companion attention note's intraday baseline: same test, scoped to the day.

`fwd0` is the session's **own** forward return - tone is fixed at admission and the price unfolds afterward, so
this needs no pairing at all and is the most direct lead test. `fwd1` / `fwd2` are the returns 1 and 2 same-day
sessions ahead.

In [ ]:
d['fwd0'] = d['sret']             # same-session forward: tone vs this session's own return (no pairing)
g = d.groupby(['symbol','day'])
d['fwd1'] = g['sret'].shift(-1)   # next same-day session's return
d['fwd2'] = g['sret'].shift(-2)   # two same-day sessions ahead
print('sessions with a next same-day session:', int(d['fwd1'].notna().sum()))

## 5. Information coefficient (Spearman rank correlation)

The **IC** is the rank correlation between tone and the forward return. If bullish tone genuinely led to gains,
we would see a clearly positive IC at a no-look-ahead horizon. We compute it for both tone measures at k=0
(same-session), k=+1 and k=+2.

In [ ]:
def spearman(a, b):
    # standard Spearman: Pearson correlation of average-tied ranks (matches scipy.stats.spearmanr,
    # and unlike an ordinal-rank shortcut it does not depend on row order when values tie)
    m = ~(np.isnan(a) | np.isnan(b)); a, b = a[m], b[m]
    if len(a) < 200:
        return float('nan'), 0
    ra = pd.Series(a).rank(method='average').values
    rb = pd.Series(b).rank(method='average').values
    return float(np.corrcoef(ra, rb)[0, 1]), len(a)

for sig in ['mean_score','pos_ratio']:
    for k, fwd in [(0,'fwd0'), (1,'fwd1'), (2,'fwd2')]:
        ic, n = spearman(d[sig].values.astype(float), d[fwd].values.astype(float))
        print(f'  IC({sig:<10}, k=+{k}) = {ic:+.4f}   (n={n:,})')

## 6. Uncertainty: a day-level block bootstrap

As in the companion note, coin-sessions within a day are cross-correlated, so we resample **whole Sundays** with
replacement (1,000 times), recompute the IC each time, and take the 2.5th-97.5th percentiles. An interval that
**straddles zero** means no detectable directional lead.

In [ ]:
def ic_ci(sig, fwd, B=1000, seed=0):
    sub = d[['day', sig, fwd]].dropna()
    days = sub['day'].values; sv = sub[sig].values.astype(float); lv = sub[fwd].values.astype(float)
    uniq = np.array(sorted(sub['day'].unique())); idx = {u: np.where(days==u)[0] for u in uniq}
    rng = np.random.default_rng(seed); D = len(uniq); out = []
    for _ in range(B):
        pick = np.concatenate([idx[uniq[i]] for i in rng.integers(0, D, D)])
        v, _ = spearman(sv[pick], lv[pick])
        if not np.isnan(v):
            out.append(v)
    return np.percentile(out, 2.5), np.percentile(out, 97.5)

for sig in ['mean_score','pos_ratio']:
    for k, fwd in [(0,'fwd0'), (1,'fwd1'), (2,'fwd2')]:
        ic, n = spearman(d[sig].values.astype(float), d[fwd].values.astype(float))
        lo, hi = ic_ci(sig, fwd)
        print(f'  IC({sig:<10}, k=+{k}) = {ic:+.4f}   [95% {lo:+.4f}, {hi:+.4f}]')

## 7. Result

On these free weekly samples every interval straddles zero: over these horizons (roughly 2-8 hours), sentiment tone carries no
detectable lead on price direction. (On the full archive the same tone measures stay within |IC| < 0.013
across the no-look-ahead horizons k = 0, +1, +2; only the tiny, *negatively*-signed same-session (k=0) and
two-ahead (k=+2) intervals exclude zero - faint mean-reversion, not a bullish lead. See the note's Section 4.) This is
a **bound**, not a proof of 'no relationship' - and it comes with one essential caveat.

**Caveat (from the note's model audit).** The pipeline's negative-tone scorer has ~64% recall and its
crypto-relevance filter drops bearish posts asymmetrically (21.7% of genuinely-bearish vs 8.8% of positive). So
`mean_score` under-represents bearish tone, and a weak bearish signal could be partially masked. The honest
reading is *no detectable directional lead at this horizon, with that measurement caveat* - not 'sentiment is
useless.' The companion note shows an attention *surge* (independent of tone) does precede higher
volatility (larger moves up and down in similar proportion).

**Going further.** A buyer of the full archive reproduces the cross-session ICs by pairing across sessions
(`groupby('symbol')` instead of `groupby(['symbol','day'])`).

Full method, results, and limitations: the research note in this repository (`research/sentiment_direction_null.md`).